In [ ]:
import numpy as np

In [ ]:
rng = np.random.default_rng(0)
d = rng.integers(0,255
                 , 1023, dtype=np.int16)
print(d[:100])
d = d.reshape((11, -1, 3))
print(d)
print(d.shape)

In [ ]:
rng = np.random.default_rng(42)

matrice = rng.integers(0,10_001, size=(4,100,100), dtype=np.int16)
print(matrice.shape)
print(matrice.nbytes)
print(matrice.size)



In [ ]:
b = matrice.argmax(axis=0)
a = (b == 3).sum()
print(matrice.mean(axis=(1,2)))
print(matrice.std(axis=(1,2)))
print(matrice.max(axis=0))
print(f"{a / b.size * 100: .1f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Matrice (4, 100, 100) avec des données aléatoires
data = np.random.rand(4, 100, 100)

# Coordonnées (Z, Y, X)
z, y, x = np.indices(data.shape)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot optimisé pour 40 000 points
sc = ax.scatter(
    x.flatten(), 
    y.flatten(), 
    z.flatten(), 
    c=data.flatten(), 
    cmap='viridis',
    s=1,          # Points très petits pour garder de la clarté
    alpha=0.7   # Transparence pour voir à travers les 4 couches
)

# Ajuster l'échelle visuelle pour que l'axe Z (4) ne soit pas écrasé par X et Y (100)
ax.set_box_aspect((100, 100, 30))

ax.set_xlabel('Largeur X (100)')
ax.set_ylabel('Hauteur Y (100)')
ax.set_zlabel('Étage Z (4)')
plt.colorbar(sc, label='Valeurs')
plt.title("Visualisation Scatter 3D d'une matrice (4, 100, 100)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Simuler une image (ex: 100x100 pixels avec 3 canaux RGB)
image = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)

# 2. Aplatir l'image pour obtenir une liste de pixels [R, G, B]
pixels = image.reshape(-1, 3)

# Normaliser les couleurs entre 0 et 1 pour Matplotlib
colors = pixels / 255.0

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# 3. Scatter map : X=Rouge, Y=Vert, Z=Bleu
ax.scatter(
    pixels[:, 0], # Rouge
    pixels[:, 1], # Vert
    pixels[:, 2], # Bleu
    c=colors,     # Chaque point prend sa propre couleur réelle
    s=5,
    alpha=0.5
)

ax.set_xlabel('Rouge (R)')
ax.set_ylabel('Vert (G)')
ax.set_zlabel('Bleu (B)')
ax.set_xlim(0, 255)
ax.set_ylim(0, 255)
ax.set_zlim(0, 255)

plt.title("Distribution des couleurs de l'image dans le cube RGB (3D)")
plt.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

data = np.random.rand(4, 100, 100)
z, y, x = np.indices(data.shape)

fig = go.Figure(data=go.Scatter3d(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    mode='markers',
    marker=dict(
        size=2,
        color=data.flatten(),
        colorscale='Viridis',
        opacity=0.2
        
    )
))

fig.update_layout(
    scene=dict(
        aspectratio=dict(x=1, y=1, z=0.3),  # Équivalent de ax.set_box_aspect
        xaxis_title='Largeur X (100)',
        yaxis_title='Hauteur Y (100)',
        zaxis_title='Étage Z (4)'
    ),
    title="Matrice 3D interactive"
)

fig.show()

In [ ]:
import numpy as np

# Génération de 3 cartes thermiques synthétiques de taille (50, 80)
# représentant 3 moments de la journée (ex: Matin=10, Midi=30, Soir=20)
np.random.seed(0)
matin = np.full((50, 80), 10) + np.random.randint(-2, 3, (50, 80))
midi  = np.full((50, 80), 30) + np.random.randint(-2, 3, (50, 80))
soir  = np.full((50, 80), 20) + np.random.randint(-2, 3, (50, 80))

In [ ]:
meteo = np.stack([matin, midi,  soir])

meteo.shape

In [ ]:
meteo_2 =  np.stack([matin, midi, soir], axis=1)
meteo_2.shape

In [ ]:
t = meteo.transpose(1,2,0)
t.shape

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.preprocessing import MinMaxScaler

# 1. CRÉATION DES DONNÉES FACTICES (Image satellite 4 bandes + 2 calques)
np.random.seed(42)

# Image satellite (4, 100, 100) -> [Bleu, Vert, Rouge, NIR]
img_satellite = np.random.randint(50, 180, (4, 100, 100))

# Calque 1 : Masque de la Forêt (Vrai au centre)
calque_foret = np.zeros((100, 100), dtype=bool)
calque_foret[20:60, 20:60] = True

# Calque 2 : Masque de l'Eau (Vrai dans le coin supérieur droit)
calque_eau = np.zeros((100, 100), dtype=bool)
calque_eau[10:40, 70:95] = True


# 2. PRÉPARATION DE L'IMAGE BACKGROUND RGB (Avéc MinMaxScaler)
# On extrait les bandes Rouge (2), Vert (1), Bleu (0) -> forme (100, 100, 3)
rgb = np.transpose(img_satellite[:3], (1, 2, 0))

# Normalisation [0, 1] avec Scikit-learn
scaler = MinMaxScaler()
rgb_normalise = scaler.fit_transform(rgb.reshape(-1, 3)).reshape(100, 100, 3)


# 3. CRÉATION DES COLORMAPS TRANSPARENTES POUR LES CALQUES
# Une colormap sur mesure qui rend le 'False' invisible (alpha=0) et le 'True' coloré (alpha=0.5)
cmap_foret = ListedColormap(['none', 'green'])
cmap_eau   = ListedColormap(['none', 'blue'])


# 4. AFFICHAGE ET SUPERPOSITION DANS MATPLOTLIB
fig, ax = plt.subplots(figsize=(8, 8))

# Couche 1 : L'image fond RGB
ax.imshow(rgb_normalise)

# Couche 2 : Superposition du calque forêt (vert transparent)
ax.imshow(calque_foret, cmap=cmap_foret, alpha=0.5)

# Couche 3 : Superposition du calque eau (bleu transparent)
ax.imshow(calque_eau, cmap=cmap_eau, alpha=0.5)

# Embellissement de la carte
ax.set_title("Superposition des Calques (Forêt & Eau) sur Image Satellite RGB", fontsize=12)
plt.axis('off')
plt.show()


In [ ]:
image= np.random.randint(0,255,(200,200,3), dtype=np.int16
                         )

In [ ]:
a = np.moveaxis(image, [2], [0])

np.round(np.mean(a, axis=(1,2)), 2)



In [ ]:
x, y , z = np.indices(np.shape(image))

In [ ]:
a = np.arange(1,5)
b = np.arange(1,10)

In [ ]:
mesh = np.meshgrid(a,b)
mesh

In [ ]:
import numpy as np
import plotly.graph_objects as go

# 1. Image 2x3 et ses axes
image = np.array([
    [100, 150, 200],
    [ 50,  80, 120]
])

x = np.array([0, 1, 2])
y = np.array([0, 1])

# 2. Génération de la grille
X, Y = np.meshgrid(x, y)

# 3. Tracé 3D des 6 points
fig = go.Figure(data=[go.Scatter3d(
    x=X.flatten(),       # [0, 1, 2, 0, 1, 2]
    y=Y.flatten(),       # [0, 0, 0, 1, 1, 1]
    z=image.flatten(),   # [100, 150, 200, 50, 80, 120]
    mode='markers+text',
    text=[f"Val: {val}" for val in image.flatten()],
    textposition="top center",
    marker=dict(
        size=10,
        color=image.flatten(),  # La couleur dépend de l'intensité (Z)
        colorscale='Viridis',
        showscale=True
    )
)])

# Configuration des axes
fig.update_layout(
    title="Les 6 pixels positionnés grâce à Meshgrid",
    scene=dict(
        xaxis=dict(title="X (Largeur / Colonnes)", tickvals=[0, 1, 2]),
        yaxis=dict(title="Y (Hauteur / Lignes)", tickvals=[0, 1]),
        zaxis=dict(title="Z (Intensité du pixel)")
    )
)

fig.show()

In [ ]:
data = np.random.rand(4, 100, 100)
z, y, x = np.indices(data.shape)

In [ ]:
n, r = np.unique_counts(z)

In [ ]:
y

In [ ]:
m = np.moveaxis(matrice, 0, -1)
shape = m.shape
print(shape)

print(np.array_equal(m[0,0], matrice[:, 0,0]))
matplot_im = m.reshape(-1,4)
print(matplot_im.shape)
matrice_depart = matplot_im.reshape(*shape).transpose((2,0,1))
print(np.array_equal(matrice, matrice_depart))
#En considérant une image sentinel sat BVR et PIR
B, V, R, PIR = matrice_depart
image_rvb = np.stack([R,V,B], axis=-1)
print(image_rvb.shape)

In [ ]:
NVDI = PIR.astype(np.float32) + R.astype(np.float32) / PIR.astype(np.float32) - R.astype(np.float32)

In [ ]:
print(NVDI)
print((NVDI < 0).sum())
print(NVDI.shape)

In [ ]:
from leafmap import Map

m = Map()
m.add_basemap("SATELLITE")
m

In [ ]:
m.save_draw_features("data/parcelle.json")

In [1]:
import geopandas as gpd
from leafmap import Map

terrain = gpd.read_file("data/parcelle.json")
utm = terrain.estimate_utm_crs()
b = terrain.to_crs(utm)
m = Map(center=[terrain.centroid.iloc[0].y,terrain.centroid.iloc[0].x], zoom=17)
m.add_basemap("SATELLITE")
m.add_gdf(b, "Terrain", dict(
    color="yellow",
    fillColor="lightgreen"
))
m

Map(center=[-2.496476772583456, 28.87598118851804], controls=(ZoomControl(options=['position', 'zoom_in_text',…

In [2]:
import numpy as np
import planetary_computer
import pystac_client

In [3]:
client = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

In [4]:
recherche = client.search(
    collections=["sentinel-2-l2a"],
    bbox=[i for i in terrain.bounds.values.reshape(-1)],
    datetime="2017-01-04/2017-12-04",
    query={"eo:cloud_cover":{"lt" : 20}}
)

items = recherche.item_collection()

In [5]:
item = min(items, key=lambda it: it.properties["eo:cloud_cover"])
print(item.assets.keys())


dict_keys(['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B09', 'B11', 'B12', 'B8A', 'SCL', 'WVP', 'visual', 'preview', 'safe-manifest', 'granule-metadata', 'inspire-metadata', 'product-metadata', 'datastrip-metadata', 'tilejson', 'rendered_preview'])


In [6]:
from odc.stac import load

images = load(
    [item],
    bands=["B02","B03","B04", "B08", "B11", "B12", "SCL"],
    bbox=[i for i in terrain.bounds.iloc[0].values.tolist()]
)

In [7]:
B4 = images.B04.isel(time=0).astype(np.float32) / 10000.0
B2=images.B02.isel(time=0).astype(np.float32) / 10000.0
B3=images.B03.isel(time=0).astype(np.float32) / 10000.0
B8 = images.B08.isel(time=0).astype(np.float32) / 10000.0

In [8]:
def diff_normalisee(a, b):
    with np.errstate(divide="ignore", invalid="ignore"):
        return (a - b) / (a + b)

ndvi = diff_normalisee(B8, B4)
ndwi = diff_normalisee(B3, B8)

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1
                    , cols=2, subplot_titles=["NDVI (Végétation)"])

fig.add_trace(
    go.Heatmap(
        z=ndvi,
        colorscale="RdYlGn",  
        zmin=-1,
        zmax=1                
    ),
    row=1, col=1
)
colorscale_custom = [
    [0.0, "black"],
    [0.5, "#3f3ff1"], 
    [1.0, "#00ffff"] 
]
fig.add_trace(
    go.Heatmap(
        z=ndwi,
        colorscale=colorscale_custom,
        zmin=-1,
        zmax=1           
    ),
    row=1, col=2
)
fig.update_yaxes(autorange='reversed')

fig.show()

In [10]:
import plotly.express as px
image_ent = np.stack([B4,B3,B2], axis=2)
px.imshow(image_ent)

In [11]:
map2 = Map(center=[terrain.centroid.iloc[0].y,terrain.centroid.iloc[0].x], zoom=17)
map2.add_basemap()
map2.add_raster(
    ndvi,
    colormap="YlGn",
    vmin=-1,
    vmax=1,
    layer_name="NDVI ODC-STAC"
)

In [12]:
map2

Map(center=[-2.492034, 28.8767365], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…